# Figure 3: Canonical Gradient and Cortical Distance

This notebook reproduces **Figure 3** from the manuscript. It assumes precomputed timecourses and connectivity matrices are available on disk, as configured via `config-results.json`.


## Config and imports

This cell loads the configuration file and imports all required libraries and utility functions.


In [ ]:
from utils import *

# Load analysis config
params = read_config('config-results.json')

# Schaefer SMC atlas
n_rois = 400
schaefer_dataset = datasets.fetch_atlas_schaefer_2018(n_rois=n_rois, resolution_mm=2)
schaefer_atlas = schaefer_dataset.maps
schaefer_labels = np.asarray(schaefer_dataset.labels, dtype=str)
smc_labels = []
for i,label in enumerate(schaefer_labels):
    if 'SomMot' in label:
        smc_labels.append(label)
smc_labels = pd.Series(smc_labels)

# Custom spinal cord atlas
sc_data = load_img(params["custom_sc_atlas"]).get_fdata()
sc_labels = open(params["custom_sc_labels"], 'r').read().splitlines()

## ROI selection and corticospinal FC

Load FC matrices, compute group-level FC, and build the SMC-spinal FC matrix used throughout Figure 3.


In [ ]:
# ------------------------------------------------------------------
# Load precomputed subject-level ROI-restricted FC matrices
# from params["save_fc_mats"] instead of recomputing FC upstream
# ------------------------------------------------------------------
fc_files = sorted(glob.glob(os.path.join(params["save_fc_mats"], "*.csv")))

if len(fc_files) == 0:
    raise FileNotFoundError(
        f"No FC csv files found in: {params['save_fc_mats']}"
    )

subFC_mats = []
sub_rois = None

for fpath in fc_files:
    df_fc = pd.read_csv(fpath, index_col=0)

    # Store ROI order from the first file and enforce consistency
    if sub_rois is None:
        sub_rois = df_fc.index.tolist()
    else:
        if df_fc.index.tolist() != sub_rois or df_fc.columns.tolist() != sub_rois:
            raise ValueError(
                f"ROI ordering mismatch in file: {fpath}"
            )

    subFC_mats.append(df_fc.values)

# Convert to array: n_subjects x n_rois x n_rois
subFC_mats = np.stack(subFC_mats, axis=0)

# ------------------------------------------------------------------
# Group-average sub-FC across subjects
# ------------------------------------------------------------------
mean_FC = np.mean(subFC_mats, axis=0)
subFC = mean_FC.copy()

# ------------------------------------------------------------------
# Rebuild ROI mapping for the already-saved FC matrices
# sub_rois should match the saved ROI-restricted FC order
# ------------------------------------------------------------------
rois_incl = sub_rois
rois_map = ['LH_SomMot' if 'LH' in roi else 'RH_SomMot' if 'RH' in roi and 'SomMot' in roi else roi.split(' ')[0]
            for roi in rois_incl]

# Identify cortical SMC indices within rois_incl
sm_idx = [j for j, roi in enumerate(rois_map) if 'SomMot' in roi]

# ------------------------------------------------------------------
# Extract cortical SMC-SMC block
# ------------------------------------------------------------------
cortical_FC = subFC[np.ix_(sm_idx, sm_idx)]

# ------------------------------------------------------------------
# Sparsify cortical FC by retaining strongest edges per row
# ------------------------------------------------------------------
sparsity_cortical = 0.9
cortical_sparse = np.array([
    row * (row > np.sort(row)[int(sparsity_cortical * len(row)) - 1])
    for row in cortical_FC
])

# ------------------------------------------------------------------
# Replace cortical block in the full subFC to obtain corticospinal FC
# ------------------------------------------------------------------
FC_cs = subFC.copy()
FC_cs[np.ix_(sm_idx, sm_idx)] = cortical_sparse

##  Computing Gradients + Clustering + Dataframes for CCA

#### Gradient Computation

In [ ]:
# SMC cortical and corticospinal gradients
grads_cortical, lambdas_cortical, FC_cortical = fit_gradients(
    FC_cs,
    [rois_incl[j] for j in sm_idx],
    rois_incl,
    n_components=5,
    approach='dm',
    kernel='spearman',
    sparsity=0,
)

grads_corticospinal, lambdas_corticospinal, FC_corticospinal = fit_gradients(
    FC_cs,
    [rois_incl[j] for j in sm_idx],
    rois_incl,
    rois_incl_y=rois_incl,
    n_components=5,
    approach='dm',
    kernel='spearman',
    sparsity=0,
)

smc_grad = grads_cortical
smc_spinal_grad = grads_corticospinal

# Align corticospinal gradients to cortical
smc_grad_norm, smc_spinal_grad_aligned, disparity = align_procrustes_matlab(
    smc_spinal_grad, smc_grad, align_dims=2
)
print(f"Disparity : grads_corticospinal, grads_cortical = {disparity:.3f}")

# Collect gradient variants
grads = {
    "SMC_Unaligned": smc_grad,
    "SMC-spinal_Unaligned": smc_spinal_grad,
    "SMC_Norm": smc_grad_norm,
    "SMC-spinal_Aligned": smc_spinal_grad_aligned,
}


#### Clustering

In [ ]:
# -----------------------
# Parameters
# -----------------------
GRAD_TYPES   = ['SMC_Norm', 'SMC-spinal_Aligned']  # which gradient sets to process
K            = 4                                   # number of clusters
RANDOM_STATE = 10                                  # for reproducibility

SMC_CLUSTER_ROIS = None   # for matching clusters across grad types


# -------------------------------------------------
# Main clustering function (both hemispheres)
# -------------------------------------------------
def cluster_and_analyze(df_all, grad_type, k=K, random_state=RANDOM_STATE,
                        cluster_feature='G2'):
    """
    1) Cluster in (Dist, cluster_feature) space using SpectralClustering
       on *both* hemispheres together.
    2) For SMC_Norm: store cluster membership.
    3) For SMC-spinal_Aligned: just use those clusters (no extra filtering).
    4) Add z-scored Dist, G1, G2 and return df_all with cluster labels.
    """
    global SMC_CLUSTER_ROIS

    x_var = 'Dist'
    y_var = cluster_feature  # 'G1' or 'G2'

    df_sub = df_all[[x_var, y_var, 'RoI', 'G1', 'G2']].dropna().copy()
    if df_sub.empty:
        print(f"No valid rows after dropna for {grad_type}")
        return None

    X = df_sub[[x_var, y_var]].values
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    spectral = SpectralClustering(
        n_clusters=k,
        affinity='rbf',
        gamma=1.0,
        random_state=random_state
    )
    labels = spectral.fit_predict(X_scaled)
    df_sub['cluster'] = labels
    sil = silhouette_score(X_scaled, labels)
    print(f"Silhouette score (k={k}): {sil:.3f}")

    # 3) Store cluster membership for SMC_Norm
    if grad_type == 'SMC_Norm':
        cluster_rois = {}
        for c in range(k):
            rois_c = df_sub.loc[df_sub['cluster'] == c, 'RoI'].tolist()
            cluster_rois[c] = rois_c
        SMC_CLUSTER_ROIS = cluster_rois

    # Map cluster labels back onto df_all
    label_map = dict(zip(df_sub['RoI'], df_sub['cluster']))
    df_all = df_all.copy()
    df_all['cluster'] = df_all['RoI'].map(label_map)
    df_all = df_all[~df_all['cluster'].isna()].copy()
    df_all['cluster'] = df_all['cluster'].astype(int)

    # 4) z-scores for Dist, G1, G2 across both hemispheres together
    df_all['z_Dist'] = (df_all['Dist'] - df_all['Dist'].mean()) / df_all['Dist'].std()
    df_all['z_G1']   = (df_all['G1']   - df_all['G1'].mean())   / df_all['G1'].std()
    df_all['z_G2']   = (df_all['G2']   - df_all['G2'].mean())   / df_all['G2'].std()

    return df_all


# Load updated trajectory tables (already filtered to desired SomMot ROIs)
lh_labels = pd.read_csv(params["dist_file_L"])
rh_labels = pd.read_csv(params["dist_file_R"])

# 1) Somatomotor labels from filtered tables, preserving LH then RH order
som_labels_L = lh_labels['roi_label'].tolist()
som_labels_R = rh_labels['roi_label'].tolist()
som_labels   = som_labels_L + som_labels_R

# 2) Distances aligned with som_labels
lh_dist = lh_labels['cum_distance_mm'].tolist()
rh_dist = rh_labels['cum_distance_mm'].tolist()
dist    = lh_dist + rh_dist

# 3) Loop over gradient types, build df_all, run clustering on both hemispheres together
df_all_dict = {}

for grad_type in GRAD_TYPES:
    g1 = grads[grad_type][:, 0]
    g2 = grads[grad_type][:, 1]

    # g1/g2 must align with som_labels; if grads is whole-brain,
    # make sure you subset grads to these somatomotor ROIs beforehand.

    df_all = pd.DataFrame({
        'G1':  g1,
        'G2':  g2,
        'RoI': som_labels,
        'Dist': dist
    })

    df_all_clustered = cluster_and_analyze(
        df_all, grad_type, k=K,
        random_state=RANDOM_STATE,
        cluster_feature='G2'
    )
    df_all_dict[grad_type] = df_all_clustered


#### Creating dataframes

In [ ]:
# ---------- 1) Prepare SMC and SMC-spinal data (both hemispheres together) ----------
df_smc  = df_all_dict['SMC_Norm'].copy()
df_spin = df_all_dict['SMC-spinal_Aligned'].copy()

# Keep needed columns and inner-join on RoI to ensure same set
df_smc  = df_smc[['RoI', 'G1', 'G2', 'z_Dist']].dropna()
df_spin = df_spin[['RoI', 'G1', 'G2', 'z_Dist']].dropna()

df_smc  = df_smc.sort_values('RoI').reset_index(drop=True)
df_spin = df_spin.sort_values('RoI').reset_index(drop=True)

common_rois = sorted(set(df_smc['RoI']) & set(df_spin['RoI']))
df_smc  = df_smc[df_smc['RoI'].isin(common_rois)].sort_values('RoI').reset_index(drop=True)
df_spin = df_spin[df_spin['RoI'].isin(common_rois)].sort_values('RoI').reset_index(drop=True)

# z-score G1,G2 in SMC space for clustering (across both hemispheres)
df_smc['z_G1'] = (df_smc['G1'] - df_smc['G1'].mean()) / df_smc['G1'].std()
df_smc['z_G2'] = (df_smc['G2'] - df_smc['G2'].mean()) / df_smc['G2'].std()

# ---------- 2) Spectral clustering in SMC G1–G2 ----------
K = 4
X_smc = df_smc[['z_G1', 'z_G2']].values
spectral = SpectralClustering(
    n_clusters=K,
    affinity='rbf',
    gamma=1.0,
    random_state=10
)
cluster_ids_smc = spectral.fit_predict(X_smc)

df_smc['cluster_smc']  = cluster_ids_smc
df_spin['cluster_smc'] = df_smc['cluster_smc'].values

# ---------- 3) Map G1–G2-based clusters back onto full SMC / spinal dfs ----------
df_all_smc    = df_all_dict['SMC_Norm'].copy()
df_all_spinal = df_all_dict['SMC-spinal_Aligned'].copy()

cluster_map_smc  = dict(zip(df_smc['RoI'],  df_smc['cluster_smc']))
cluster_map_spin = dict(zip(df_spin['RoI'], df_spin['cluster_smc']))

df_all_smc['cluster_smc']    = df_all_smc['RoI'].map(cluster_map_smc)
df_all_spinal['cluster_smc'] = df_all_spinal['RoI'].map(cluster_map_spin)

# Drop RoIs without a cluster, cast to int
df_all_smc    = df_all_smc.dropna(subset=['cluster_smc']).copy()
df_all_spinal = df_all_spinal.dropna(subset=['cluster_smc']).copy()
df_all_smc['cluster_smc']    = df_all_smc['cluster_smc'].astype(int)
df_all_spinal['cluster_smc'] = df_all_spinal['cluster_smc'].astype(int)

# Recompute z-scores within each df (across both hemispheres)
for df in [df_all_smc, df_all_spinal]:
    df['z_G1'] = (df['G1'] - df['G1'].mean()) / df['G1'].std()
    df['z_G2'] = (df['G2'] - df['G2'].mean()) / df['G2'].std()

# Hemisphere label
for df in [df_all_smc, df_all_spinal]:
    df['Hemisphere'] = df['RoI'].str.contains('RH').map({True: 'RH', False: 'LH'})


## Panel A – Canonical Gradient Analysis

CCA, grad–distance visualization, and stability/null. Please note the functional manifold inset of the cortical (smc-smc) gradient space is same as previously computed in Fig.1 and Fig.2. The cortical render of the cortical distance are visulaized in Brainspace using a CSV file storing the cortical distance measure (computed as nearest neighbour path from medial to dorsal site on the SMC cortical strip) for each of the "SomMot" labels. 

In [ ]:
# ---------------------------------------------------------------------
# 0. Reduced dataframe builder
# ---------------------------------------------------------------------

def build_reduced_df(df):
    """
    Adapt column names here if needed.
    Expected columns in df:
      'RoI', 'z_G1', 'z_G2', 'z_Dist', 'Hemisphere'
      and a cluster column: 'cluster_smc' or 'cluster_spinal'
    """
    cols = ["RoI", "z_G1", "z_G2", "z_Dist", "Hemisphere"]
    # cluster column stays in df but not used directly by CCA
    return df[cols + [c for c in df.columns if c.startswith("cluster_")]].copy()


df_smc_red    = build_reduced_df(df_all_smc)
df_spinal_red = build_reduced_df(df_all_spinal)


# ---------------------------------------------------------------------
# 1. Core CCA + permutation functions
# ---------------------------------------------------------------------

def run_cca(df, n_components=1):
    X = df[["z_G1", "z_G2"]].values
    Y = df[["z_Dist"]].values

    cca = CCA(n_components=n_components)
    cca.fit(X, Y)
    X_c, Y_c = cca.transform(X, Y)

    x1 = X_c[:, 0]
    y1 = Y_c[:, 0]
    can_corr = np.corrcoef(x1, y1)[0, 1]

    return cca, X_c, Y_c, can_corr


def cca_permutation_test(df, n_perm=5000, random_state=0):
    rng = check_random_state(random_state)

    _, _, _, can_corr_true = run_cca(df)

    perm_corrs = np.zeros(n_perm)
    for i in range(n_perm):
        df_perm = df.copy()
        df_perm["z_Dist"] = rng.permutation(df_perm["z_Dist"].values)
        _, _, _, perm_corrs[i] = run_cca(df_perm)

    p_val = (np.sum(np.abs(perm_corrs) >= np.abs(can_corr_true)) + 1) / (n_perm + 1)
    return can_corr_true, perm_corrs, p_val


def run_cca_with_null(df, label, n_perm=5000, random_state=0):
    cca, X_c, Y_c, can_corr = run_cca(df)
    can_corr_true, perm_corrs, p_val = cca_permutation_test(
        df, n_perm=n_perm, random_state=random_state
    )

    df_out = df.copy()
    df_out["G_can"] = X_c[:, 0]
    df_out["Dist_can"] = Y_c[:, 0]

    print(f"\n=== {label} ===")
    print(f"n ROIs = {len(df_out)}")
    print(f"Canonical corr (G1,G2 vs Dist): {can_corr_true:.3f}")
    print(f"Permutation p-value: {p_val:.4f}")

    return df_out, can_corr_true, p_val, perm_corrs


# ---------------------------------------------------------------------
# 2. Run for SMC / spinal and hemispheres, and build summary table
# ---------------------------------------------------------------------

def run_all_cca(
    df_smc,
    df_spinal,
    n_perm=5000,
    random_state=0,
    do_hemispheres=True,
):
    results = {}
    summary_rows = []

    def add_summary_row(system, hemi_label, r, p, n):
        summary_rows.append(
            {"System": system, "Hemisphere": hemi_label,
             "n_ROI": n, "canonical_r": r, "p_perm": p}
        )

    # SMC all hemispheres
    df_smc_all, r_smc_all, p_smc_all, perm_smc_all = run_cca_with_null(
        df_smc, label="SMC (LH+RH)", n_perm=n_perm, random_state=random_state
    )
    results["SMC_all"] = (df_smc_all, r_smc_all, p_smc_all, perm_smc_all)
    add_summary_row("SMC", "LH+RH", r_smc_all, p_smc_all, len(df_smc_all))

    # Spinal all hemispheres
    df_sp_all, r_sp_all, p_sp_all, perm_sp_all = run_cca_with_null(
        df_spinal, label="Spinal (LH+RH)", n_perm=n_perm, random_state=random_state
    )
    results["Spinal_all"] = (df_sp_all, r_sp_all, p_sp_all, perm_sp_all)
    add_summary_row("Spinal", "LH+RH", r_sp_all, p_sp_all, len(df_sp_all))

    if do_hemispheres:
        # SMC LH
        df_smc_lh = df_smc[df_smc["Hemisphere"] == "LH"]
        if len(df_smc_lh) > 2:
            df_smc_lh_out, r_smc_lh, p_smc_lh, perm_smc_lh = run_cca_with_null(
                df_smc_lh, label="SMC LH", n_perm=n_perm, random_state=random_state
            )
            results["SMC_LH"] = (df_smc_lh_out, r_smc_lh, p_smc_lh, perm_smc_lh)
            add_summary_row("SMC", "LH", r_smc_lh, p_smc_lh, len(df_smc_lh_out))

        # SMC RH
        df_smc_rh = df_smc[df_smc["Hemisphere"] == "RH"]
        if len(df_smc_rh) > 2:
            df_smc_rh_out, r_smc_rh, p_smc_rh, perm_smc_rh = run_cca_with_null(
                df_smc_rh, label="SMC RH", n_perm=n_perm, random_state=random_state
            )
            results["SMC_RH"] = (df_smc_rh_out, r_smc_rh, p_smc_rh, perm_smc_rh)
            add_summary_row("SMC", "RH", r_smc_rh, p_smc_rh, len(df_smc_rh_out))

        # Spinal LH
        df_sp_lh = df_spinal[df_spinal["Hemisphere"] == "LH"]
        if len(df_sp_lh) > 2:
            df_sp_lh_out, r_sp_lh, p_sp_lh, perm_sp_lh = run_cca_with_null(
                df_sp_lh, label="Spinal LH", n_perm=n_perm, random_state=random_state
            )
            results["Spinal_LH"] = (df_sp_lh_out, r_sp_lh, p_sp_lh, perm_sp_lh)
            add_summary_row("Spinal", "LH", r_sp_lh, p_sp_lh, len(df_sp_lh_out))

        # Spinal RH
        df_sp_rh = df_spinal[df_spinal["Hemisphere"] == "RH"]
        if len(df_sp_rh) > 2:
            df_sp_rh_out, r_sp_rh, p_sp_rh, perm_sp_rh = run_cca_with_null(
                df_sp_rh, label="Spinal RH", n_perm=n_perm, random_state=random_state
            )
            results["Spinal_RH"] = (df_sp_rh_out, r_sp_rh, p_sp_rh, perm_sp_rh)
            add_summary_row("Spinal", "RH", r_sp_rh, p_sp_rh, len(df_sp_rh_out))

    summary_df = pd.DataFrame(summary_rows)
    return results, summary_df


# Run once
results, summary_df = run_all_cca(df_smc_red, df_spinal_red, n_perm=5000, random_state=42)

print("\nCCA summary table:")
print(summary_df)


## Panel B-F – Canonical Gradient vs. Cortical Distance

Canonical gradient from CCA of G1 and G2 against cortical distance, shown as a scatter with linear regression fit and as a kernel density contour plot

In [ ]:
cluster_hex = {
    0: "#219ebc",
    1: "#023047",
    2: "#ffb703",
    3: "#fb8500",
}


def plot_cluster_reg_panels_cca_single(df, cluster_col, fig_title_prefix):
    """
    Generate:
      - One overview panel: G_can vs z_Dist colored by cluster
      - One panel per cluster: regression of G_can vs z_Dist (LH+RH pooled)
    df is expected to contain: 'z_Dist', 'G_can', and cluster_col.
    """

    # Unique cluster labels (robust to dtype)
    clusters = sorted(df[cluster_col].dropna().astype(int).unique())

    # -------- Overview figure --------
    fig, ax = plt.subplots(figsize=(5, 5), dpi=300)
    sns.scatterplot(
        ax=ax,
        data=df,
        x="z_Dist",
        y="G_can",
        hue=cluster_col,
        alpha=0.7,
        palette=cluster_hex,
        s=200,
        edgecolor="k",
        legend=False,
    )
    ax.set_title(f"{fig_title_prefix}\nG_can vs z(Dist) by cluster")
    ax.set_xlabel("z(Dist)")
    ax.set_ylabel("Canonical gradient (G_can)")
    plt.tight_layout()
    plt.show()

    # -------- Per-cluster regression panels --------
    for c in clusters:
        df_c = df[df[cluster_col] == c].copy()
        if df_c.empty:
            continue

        X = df_c["z_Dist"].values.reshape(-1, 1)
        y = df_c["G_can"].values

        if len(df_c) >= 2:
            reg = LinearRegression()
            reg.fit(X, y)
            y_pred = reg.predict(X)
            slope = reg.coef_[0]
            r2 = r2_score(y, y_pred)
            r, p_corr = stats.pearsonr(df_c["z_Dist"].values, df_c["G_can"].values)
        else:
            slope, r2, r, p_corr = np.nan, np.nan, np.nan, np.nan

        base_color = cluster_hex.get(int(c), "#000000")

        fig, ax = plt.subplots(figsize=(5, 5), dpi=300)
        sns.regplot(
            ax=ax,
            data=df_c,
            x="z_Dist",
            y="G_can",
            scatter_kws={"s": 200, "edgecolor": "k"},
            line_kws={"color": base_color, "linewidth": 1.5},
            ci=95,
            color=base_color,
        )
        ax.set_xlabel("z(Dist)")
        ax.set_ylabel("Canonical gradient (G_can)")
        ax.set_title(
            f"{fig_title_prefix} | Cluster {int(c)}\n"
            f"slope={slope:.3f}, R²={r2:.3f},  p={p_corr:.4f} (n={len(df_c)})"
        )
        plt.tight_layout()
        plt.show()


# ----------------------------
# Calls for SMC and SMC–spinal
# ----------------------------

# SMC (LH+RH): df_smc_best from results["SMC_all"]
df_smc_best = results["SMC_all"][0]  # contains G_can, z_Dist, cluster_smc
plot_cluster_reg_panels_cca_single(
    df=df_smc_best,
    cluster_col="cluster_smc",
    fig_title_prefix="SMC CCA (LH+RH)",
)

# Spinal (LH+RH) if needed:
df_spinal_best = results["Spinal_all"][0]
plot_cluster_reg_panels_cca_single(
    df=df_spinal_best,
    cluster_col="cluster_smc",
    fig_title_prefix="SMC-Spinal CCA (LH+RH)",
)